In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

debug = True

if debug:
    bronze_catalog = "bronze"
    bronze_schema = "sistema_leitos"
    bronze_table = "leitos"
    silver_catalog = "silver"
    silver_schema = "sistema_leitos"
else:
    bronze_catalog = dbutils.widgets.get("bronze_catalog")
    bronze_schema = dbutils.widgets.get("bronze_schema")
    bronze_table = dbutils.widgets.get("bronze_table")
    silver_catalog = dbutils.widgets.get("silver_catalog")
    silver_schema = dbutils.widgets.get("silver_schema")

bronze_table_full_name = f"{bronze_catalog}.{bronze_schema}.{bronze_table}"
checkpoint_path = f"Volumes/raw-data/sistema_leitos/leitos/silver/{bronze_table}_checkpoint/"

In [ ]:
def process_silver_upserts(micro_batch_df, batch_id):
    print(f"Iniciando micro-lote {batch_id}...")

    changes_df = micro_batch_df.filter(F.col("_change_type").isin(["insert", "update_postimage"])).cache()

    if changes_df.isEmpty():
        print(f"Micro-lote {batch_id} não contém novas inserções ou atualizações. Finalizando")
        return

    dim_tempo_target = f"{silver_catalog}.{silver_schema}.dim_tempo"
    dim_tempo_updates = (
        changes_df.select(F.col("comp")).distinct()
        .withColumn("id_tempo", F.to_date(F.col("comp").cast("string"), "yyyyMM"))
        .select(
            F.col("id_tempo"), F.col("comp").alias("ano_mes"), F.year("id_tempo").alias("ano"),
            F.month("id_tempo").alias("mes"), F.quarter("id_tempo").alias("trimestre"),
            F.when(F.month("id_tempo") <= 6, 1).otherwise(2).alias("semestre")
        )
    )
    DeltaTable.forName(spark, dim_tempo_target).alias("target").merge(
        dim_tempo_updates.alias("source"),
        "target.id_tempo = source.id_tempo"
    ).whenNotMatchedInsertAll().execute()

    dim_est_target = f"{silver_catalog}.{silver_schema}.dim_estabelecimento"
    window_latest = Window.partitionBy("cnes").orderBy(F.col("_commit_version").desc())
    
    est_updates = (
        changes_df.withColumn("rn", F.row_number().over(window_latest)).filter(F.col("rn") == 1).drop("rn")
        .select(
            F.col("cnes").cast("string").alias("cnes"), F.trim(F.upper(F.col("nome_estabelecimento"))).alias("nome_estabelecimento"),
            F.trim(F.upper(F.col("razao_social"))).alias("razao_social"), F.trim(F.upper(F.col("regiao"))).alias("regiao"),
            F.trim(F.upper(F.col("uf"))).alias("uf"), F.trim(F.upper(F.col("municipio"))).alias("municipio"),
            F.col("co_cep").cast("string").alias("cep"), F.col("natureza_juridica").cast("string").alias("cod_natureza_juridica"),
            F.trim(F.upper(F.col("desc_natureza_juridica"))).alias("desc_natureza_juridica"), F.trim(F.upper(F.col("tp_gestao"))).alias("tipo_gestao"),
            F.col("updated_at").alias("data_ultima_atualizacao_bronze")
        )
    )
    DeltaTable.forName(spark, dim_est_target).alias("target").merge(
        est_updates.alias("source"), "target.cnes = source.cnes"
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

    fact_leitos_target = f"{silver_catalog}.{silver_schema}.fact_leitos"
    metric_cols = ['leitos_existente', 'leitos_sus', 'uti_total_exist', 'uti_total_sus', 'uti_adulto_exist', 'uti_adulto_sus', 'uti_pediatrico_exist', 'uti_pediatrico_sus', 'uti_neonatal_exist', 'uti_neonatal_sus', 'uti_queimado_exist', 'uti_queimado_sus', 'uti_coronariana_exist', 'uti_coronariana_sus']
    
    fact_updates = (
        changes_df.fillna(0, subset=metric_cols)
        .select(
            F.to_date(F.col("comp").cast("string"), "yyyyMM").alias("id_tempo"), F.col("cnes").cast("string").alias("cnes"),
            F.trim(F.upper(F.col("motivo_desabilitacao"))).alias("motivo_desabilitacao"), F.col("leitos_existente").alias("qtd_leitos_existentes"),
            F.col("leitos_sus").alias("qtd_leitos_sus"), F.col("uti_total_exist").alias("qtd_uti_total_exist"),
            F.col("uti_total_sus").alias("qtd_uti_total_sus"), F.col("uti_adulto_exist").alias("qtd_uti_adulto_exist"),
            F.col("uti_adulto_sus").alias("qtd_uti_adulto_sus"), F.col("uti_pediatrico_exist").alias("qtd_uti_pediatrico_exist"),
            F.col("uti_pediatrico_sus").alias("qtd_uti_pediatrico_sus"), F.col("uti_neonatal_exist").alias("qtd_uti_neonatal_exist"),
            F.col("uti_neonatal_sus").alias("qtd_uti_neonatal_sus"), F.col("uti_queimado_exist").alias("qtd_uti_queimado_exist"),
            F.col("uti_queimado_sus").alias("qtd_uti_queimado_sus"), F.col("uti_coronariana_exist").alias("qtd_uti_coronariana_exist"),
            F.col("uti_coronariana_sus").alias("qtd_uti_coronariana_sus"), F.col("updated_at").alias("data_atualizacao_bronze")
        )
    )
    DeltaTable.forName(spark, fact_leitos_target).alias("target").merge(
        fact_updates.alias("source"), "target.id_tempo = source.id_tempo AND target.cnes = source.cnes"
    ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

    changes_df.unpersist()
    print(f"Micro-lote {batch_id} processado com sucesso.")

In [ ]:
print("Iniciando stream da camada Prata...")

bronze_cdf_stream = (
    spark.readStream
    .option("readChangeFeed", "true")
    .table(bronze_table_full_name)
)

query = (
    bronze_cdf_stream.writeStream
    .foreachBatch(process_silver_upserts)
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True) 
    .start()
)

print("Stream da camada Prata finalizado.")